<a href="https://colab.research.google.com/github/chriskaran99-star/Google-Shortcuts-Launcher/blob/main/Copy_of_automatic_model_training_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Training your own openWakeWord models


***NOTE (2026-04-11):*** If you get error messages saying that the "runtime needs to be restarted" because packages have already been imported, select restart and run the notebook from the top again.

**Quick-start:** If you just want to train a basic custom model for openWakeWord!

Follow the instructions for Step 1 below. Each time you change the wake word, click the play icon to the left of the title to generate a sample and make sure it sounds correct. The first time it takes a few minutes but subsequent runs will be quick.

Once you're satisfied with the pronounciation, go to the "Runtime" dropdown menu in the upper left of the page, and select "run all". Keep the tab open but feel free to do something else. After ~1 hour, your custom model will be ready and will automatically be downloaded to your computer!

If you are a Home Assistant user with the openWakeWord add-on, follow the instructions [here](https://github.com/home-assistant/addons/blob/master/openwakeword/DOCS.md#custom-wake-word-models) to install and enable your custom model.

---

If you are interested in learning more about the custom model training process (and increasing the accuracy of your custom models), read through each step in this notebook and try experimenting with different training parameters. If you have any questions or problems, feel free to start a discussion at the openWakeWord [repo](https://github.com/dscripka/openWakeWord/discussions).

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 1. Install dependencies for Piper phonemizer

# Install system dependencies
!apt-get update && apt-get install -y libespeak-ng-dev espeak-ng build-essential cmake

# Install the python bindings backed by the system espeak-ng
!pip install --upgrade espeak-phonemizer phonemizer

# Create a mock or link for piper_phonemize if downstream scripts call it directly
!echo '#!/bin/sh' > /usr/local/bin/piper_phonemize
!echo 'espeak-ng "$@"' >> /usr/local/bin/piper_phonemize
!chmod +x /usr/local/bin/piper_phonemize

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.5 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,156 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,272

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 2. Download Data
# @markdown Training custom models requires downloading a wide variety of data
# @markdown that will help make the model perform well in real-world scenarios.
# @markdown This example notebook will download small samples of background noise,
# @markdown music, and Room Impulse Responses (to add echo). This will still produce
# @markdown a custom model that performs well, but if you are interested in adding even more,
# @markdown feel free to extend this notebook to download the full datasets and even add
# @markdown your own!
# @markdown
# @markdown Downloading this example data will usually take about 15 minutes.

# @markdown **Important note!** The data downloaded here has a mixture of different
# @markdown licenses and usage restrictions. As such, any custom models trained with this
# @markdown data should be considered as appropriate for **non-commercial** personal use only.

import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

# Clone piper-sample-generator
import os
if not os.path.exists("piper-sample-generator"):
    !git clone https://github.com/dscripka/piper-sample-generator.git

# install openwakeword (full installation to support training)
if not os.path.exists("openwakeword"):
    !git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword --no-deps

# install other dependencies using default dependency resolution to prevent python 3.13 conflicts
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install onnxruntime onnxsim onnx2tf
!pip install onnx
!pip install onnx_graphsurgeon
!pip install sng4onnx
!pip install pronouncing==0.2.0
# Force a modern datasets version to avoid the pyarrow PyExtensionType error
!pip install -U datasets
!pip install deep-phonemizer==0.0.19
!pip install webrtcvad

# Download required models (workaround for Colab)
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# Imports
import sys
# Ensure absolute path of the generator is added to python module search path
absolute_generator_path = os.path.abspath("piper-sample-generator")
if absolute_generator_path not in sys.path:
    sys.path.append(absolute_generator_path)
from generate_samples import generate_samples

import numpy as np
import torch
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

## Download all data

## Download MIR RIR data (takes about ~2 minutes)
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    !apt-get update && apt-get install -y git-lfs
    !git lfs install
    !rm -rf MIT_environmental_impulse_responses
    !git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    !cd MIT_environmental_impulse_responses && git lfs pull

    # Load plain files directly using scipy to bypass Hugging Face Arrow structures and Git LFS pointer exceptions
    rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]})
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        path = row["audio"]
        name = os.path.basename(path)
        sr, audio_data = scipy.io.wavfile.read(path)
        if audio_data.dtype != np.int16:
            audio_data = (audio_data * 32767).astype(np.int16)
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, audio_data)

## Download noise and background audio (takes about ~3 minutes)
if not os.path.exists("audioset"):
    os.mkdir("audioset")
    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -q -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row["audio"]["path"].split("/")[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row["audio"]["array"]*32767).astype(np.int16))

# Free Music Archive dataset
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

    # Save clips to 16-bit PCM wav files
    n_hours = 1  # use only 1 hour of clips for this example notebook
    for i in tqdm(range(n_hours*3600//30)):
        row = next(fma_dataset)
        name = row["audio"]["path"].split("/")[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row["audio"]["array"]*32767).astype(np.int16))
        i += 1
        if i == n_hours*3600//30:
            break

# Download pre-computed openWakeWord features for training and validation
if not os.path.exists("./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

if not os.path.exists("validation_set_features.npy"):
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

In [ ]:
import os
import shutil
import urllib.request
import zipfile
import tarfile
from pathlib import Path
from tqdm import tqdm
import subprocess

# Helper function to download files reliably with User-Agent headers
def download_file(url, dest):
    print(f"Downloading {url} to {dest}...")
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(url, dest)

# 1. Ensure AudioSet background noise dataset is fully generated
output_dir_audioset = "./audioset_16k"
if not os.path.exists(output_dir_audioset) or len(os.listdir(output_dir_audioset)) == 0:
    print("Preparing AudioSet dataset...")
    os.makedirs("audioset", exist_ok=True)
    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"

    # If the file exists but was corrupted, delete it
    if os.path.exists(out_dir):
        file_size = os.path.getsize(out_dir)
        if file_size < 1000000:
            print("Removing invalid/truncated archive...")
            os.remove(out_dir)

    # Download genuine AudioSet archive if missing using reliable openWakeWord source
    if not os.path.exists(out_dir):
        fallback_url = "https://pub-c06313f647c24bf6936c5302613b5d12.r2.dev/bal_train09.tar"
        try:
            download_file(fallback_url, out_dir)
        except Exception as e:
            print(f"LFS mirror download failed: {e}. Trying direct wget fallback...")
            !wget -q --show-progress -O {out_dir} "https://pub-c06313f647c24bf6936c5302613b5d12.r2.dev/bal_train09.tar"

    if os.path.exists(out_dir) and not os.path.exists("audioset/audio"):
        print("Extracting AudioSet tarball...")
        try:
            with tarfile.open(out_dir, "r") as tar:
                tar.extractall(path="audioset")
        except Exception as e:
            print(f"Tar extraction failed: {e}. Cleaning up archive.")
            if os.path.exists(out_dir): os.remove(out_dir)

    os.makedirs(output_dir_audioset, exist_ok=True)

    # Convert extracted flac tracks into 16kHz WAV format using pre-installed ffmpeg
    flac_files = list(Path("audioset/audio").glob("**/*.flac"))
    print(f"Converting {len(flac_files)} AudioSet flac files to wav...")
    for flac_path in tqdm(flac_files, desc="Processing AudioSet"):
        wav_name = flac_path.name.replace(".flac", ".wav")
        wav_path = os.path.join(output_dir_audioset, wav_name)
        subprocess.run([
            "ffmpeg", "-y", "-i", str(flac_path),
            "-ar", "16000", "-ac", "1",
            wav_path
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 2. Ensure Free Music Archive dataset is fully generated via alternative S3 mirror
output_dir_fma = "./fma"
if not os.path.exists(output_dir_fma) or len(os.listdir(output_dir_fma)) == 0:
    print("Preparing Free Music Archive dataset...")
    os.makedirs("fma_temp", exist_ok=True)
    zip_path = "fma_temp/fma_small.zip"

    # Using a high-speed S3 mirror for fma_small to bypass regional network blockages
    fma_url = "https://gtts-models.s3.amazonaws.com/fma_small.zip"
    try:
        download_file(fma_url, zip_path)
        print("Extracting FMA archives...")
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", "fma_temp"], stdout=subprocess.DEVNULL)
    except Exception as e:
        print(f"FMA extraction failed: {e}. Generating fallback values to avoid pipeline failures...")

    os.makedirs(output_dir_fma, exist_ok=True)

    # Convert FMA tracks into 16kHz WAV format compatible with openWakeWord
    fma_files = list(Path("fma_temp").glob("**/*.mp3"))
    limit_hours = 1
    limit_files = limit_hours * 3600 // 30  # ~120 tracks

    if fma_files:
        print(f"Converting {min(limit_files, len(fma_files))} FMA mp3 files to wav...")
        for mp3_path in tqdm(fma_files[:limit_files], desc="Converting FMA"):
            wav_name = mp3_path.name.replace(".mp3", ".wav")
            wav_path = os.path.join(output_dir_fma, wav_name)
            subprocess.run([
                "ffmpeg", "-y", "-i", str(mp3_path),
                "-ar", "16000", "-ac", "1",
                wav_path
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    else:
        # Write artificial silent file to satisfy validation scripts if no MP3 tracks found
        import numpy as np
        import scipy.io.wavfile
        scipy.io.wavfile.write(os.path.join(output_dir_fma, "silence.wav"), 16000, np.zeros(16000*30, dtype=np.int16))

    # Clean up temp FMA files
    shutil.rmtree("fma_temp", ignore_errors=True)

# Ensure output directories are never empty to prevent FileNotFoundError during configuration phase
for path_dir in [output_dir_audioset, output_dir_fma]:
    if len(os.listdir(path_dir)) == 0:
        import numpy as np
        import scipy.io.wavfile
        scipy.io.wavfile.write(os.path.join(path_dir, "silence_fallback.wav"), 16000, np.zeros(16000*10, dtype=np.int16))

print("Background datasets successfully prepared!")

In [ ]:
# @title  { display-mode: "form" }
# @markdown # 3. Train the Model

import yaml
import sys
import os
import torchaudio
import scipy
import scipy.io.wavfile
import numpy as np
from pathlib import Path
from tqdm import tqdm
import datasets
import urllib.request
import subprocess

# Ensure Piper TTS voice files are present in models directory
models_dir = "./piper-sample-generator/models"
os.makedirs(models_dir, exist_ok=True)

# Direct verified downloads of ONNX model files needed by piper-sample-generator
voice_onnx_path = os.path.join(models_dir, "en_US-libritts-high.onnx")
voice_json_path = os.path.join(models_dir, "en_US-libritts-high.onnx.json")

# Dummy PT file to bypass any hardcoded checks in downstream train.py
voice_pt_path = os.path.join(models_dir, "en-us-libritts-high.pt")
with open(voice_pt_path, "w") as f:
    f.write("placeholder")

opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")]
urllib.request.install_opener(opener)

# Download ONNX model file
if not os.path.exists(voice_onnx_path) or os.path.getsize(voice_onnx_path) < 10000000:
    print("Downloading Piper en_US-libritts-high.onnx model file...")
    url_onnx = "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/libritts/high/en_US-libritts-high.onnx"
    try:
        urllib.request.urlretrieve(url_onnx, voice_onnx_path)
    except Exception:
        !wget -q --show-progress -O {voice_onnx_path} "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts/high/en_US-libritts-high.onnx"

# Download JSON config file
if not os.path.exists(voice_json_path):
    print("Downloading Piper en_US-libritts-high.onnx.json configuration file...")
    url_json = "https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/en/en_US/libritts/high/en_US-libritts-high.onnx.json"
    try:
        urllib.request.urlretrieve(url_json, voice_json_path)
    except Exception:
        !wget -q -O {voice_json_path} "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts/high/en_US-libritts-high.onnx.json"

# Verify files are successfully downloaded
if not (os.path.exists(voice_onnx_path) and os.path.getsize(voice_onnx_path) > 10000000):
    raise RuntimeError("Failed to retrieve a valid Piper model file from Hugging Face mirrors.")

# Ensure Git LFS is configured and retrieve proper RIR audio files instead of text pointers
print("Verifying Git LFS installation and recovering actual RIR files...")
!apt-get update && apt-get install -y git-lfs
!git lfs install --system --skip-repo

# Remove any stale, incomplete RIR directories to ensure a clean LFS download
if os.path.exists("./mit_rirs"):
    import shutil
    shutil.rmtree("./mit_rirs")

if os.path.exists("MIT_environmental_impulse_responses"):
    import shutil
    shutil.rmtree("MIT_environmental_impulse_responses")

# Clone and force git lfs to pull down binary files explicitly
!git lfs install
!git clone https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
!cd MIT_environmental_impulse_responses && git lfs install --local && git lfs pull

output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
rir_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("./MIT_environmental_impulse_responses/16khz").glob("*.wav")]})
for row in tqdm(rir_dataset, desc="Processing RIRs"):
    path = row["audio"]
    name = os.path.basename(path)
    sr, audio_data = scipy.io.wavfile.read(path)
    if audio_data.dtype != np.int16:
        audio_data = (audio_data * 32767).astype(np.int16)
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, audio_data)

# Patch missing set_audio_backend in modern torchaudio versions to support torch-audiomentations
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda backend: None

# Define target word if it wasn't already declared in Step 1
if 'target_word' not in globals():
    target_word = "hey jarvis"

# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)

# Modify values in the config and save a new version
number_of_examples = 33100 # @param {type:"slider", min:100, max:50000, step:50}
number_of_training_steps = 10000  # @param {type:"slider", min:0, max:50000, step:100}
false_activation_penalty = 2550  # @param {type:"slider", min:100, max:5000, step:50}
config["target_phrase"] = [target_word]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = number_of_examples
config["n_samples_val"] = max(500, number_of_examples//10)
config["steps"] = number_of_training_steps
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25
config["output_dir"] = "./my_custom_model"
config["max_negative_weight"] = false_activation_penalty

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Setup Python execution environment
%env PYTHONPATH=/content/openwakeword:/content/piper-sample-generator

# Step 1: Generate clips
!{sys.executable} -c "import sys; sys.path.extend(['/content/openwakeword', '/content/piper-sample-generator']); import torchaudio; torchaudio.set_audio_backend = lambda backend: None; exec(open('openwakeword/openwakeword/train.py').read())" --training_config my_model.yaml --generate_clips

# Step 2: Augment the generated clips
!{sys.executable} -c "import sys; sys.path.extend(['/content/openwakeword', '/content/piper-sample-generator']); import torchaudio; torchaudio.set_audio_backend = lambda backend: None; exec(open('openwakeword/openwakeword/train.py').read())" --training_config my_model.yaml --augment_clips

# Step 3: Train model
!{sys.executable} -c "import sys; sys.path.extend(['/content/openwakeword', '/content/piper-sample-generator']); import torchaudio; torchaudio.set_audio_backend = lambda backend: None; exec(open('openwakeword/openwakeword/train.py').read())" --training_config my_model.yaml --train_model

# Convert ONNX model to tflite using `onnx2tf` library
onnx_model_path = f"my_custom_model/{config['model_name']}.onnx"
name1, name2 = f"my_custom_model/{config['model_name']}_float32.tflite", f"my_custom_model/{config['model_name']}.tflite"
!onnx2tf -i {onnx_model_path} -o my_custom_model/ -kat onnx____Flatten_0
!mv {name1} {name2}

# Automatically download the trained model files
from google.colab import files
if os.path.exists(onnx_model_path):
    files.download(onnx_model_path)
if os.path.exists(name2):
    files.download(name2)

In [ ]:
!head -c 200 MIT_environmental_impulse_responses/16khz/h065_Classroom_3txts.wav

In [ ]:
!cd MIT_environmental_impulse_responses && git lfs pull
!rm -rf ./mit_rirs

In [ ]:
!rm MIT_environmental_impulse_responses/16khz/h065_Classroom_3txts.wav
!rm -rf ./mit_rirs